# C3 - Day 4: Decision Tree and Random Forest
**AnchorAI | Component 3: Cognitive Checkpointing**

Goal: check whether non-linear models predict resumption difficulty better than the Day 3 Logistic Regression baseline, and choose the final Model 1 **fairly**.

Fairness rules (same as Day 3):
* all models use the **same participant-wise folds** and the **same 30 test participants**;
* hyperparameters are tuned with cross-validation on the **training** participants only;
* the test participants are used **once**, to report results, never to choose settings.

> Run on **CPU runtime**. Needs `samples.csv` and `meta.json` from Day 2. The tuning step takes a few minutes.

## Step 0 - Mount Drive, load the Day 2 data

In [ ]:
import os, csv, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/C3'
    print('Running in Colab - Drive mounted.')
except ImportError:
    BASE = os.environ.get('C3_BASE', r'C:\Users\user\OneDrive\Documents\AnchorAI\AnchorAI\components\c3_checkpointing')
    print('Running locally.')

DATA_DIR   = os.path.join(BASE, 'data', 'raw', 'hirsch')
PROC_DIR   = os.path.join(BASE, 'data', 'processed')
OUTPUT_DIR = os.path.join(BASE, 'outputs')
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, classification_report, ConfusionMatrixDisplay
import joblib

MODELS_DIR = os.path.join(BASE, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

S = pd.read_csv(os.path.join(PROC_DIR, 'samples.csv'))
with open(os.path.join(PROC_DIR, 'meta.json')) as f:
    meta = json.load(f)
LABEL_NAMES = meta['label_names']                  # 0 = Low, 1 = Medium, 2 = High

train = S[S['split'] == 'train'].reset_index(drop=True)
test  = S[S['split'] == 'test'].reset_index(drop=True)
print(f'Train: {len(train):,} trials, {train.pp_id.nunique()} participants | Test: {len(test):,} trials, {test.pp_id.nunique()} participants')

## Step 1 - Models and helper functions
Four models are compared:
* **Dummy**: always predicts the most common class (the "no learning" reference).
* **Logistic Regression**: the Day 3 baseline (a straight-line decision rule).
* **Decision Tree**: a flowchart of yes/no questions on the features, e.g. "was the interruption task longer than 2.5 s?". Easy to read, but a deep tree memorises the training participants (**overfitting**).
* **Random Forest**: many trees trained on random parts of the data; their votes are averaged. Usually more stable than one tree.

Trees do not need feature scaling, so only `experiment` is one-hot encoded.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.inspection import permutation_importance

ALL_FEATURES = ['sequence', 'block', 'trial', 'experiment',
                'interruption_rt', 'interruption_error',
                'rt_subtask1', 'rt_subtask2', 'pre_errors',
                'pp_speed']

def prep_scaled(features):
    return ColumnTransformer([('num', StandardScaler(), [c for c in features if c != 'experiment']),
                              ('cat', OneHotEncoder(handle_unknown='ignore'), ['experiment'])])

def prep_trees(features):
    return ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['experiment']),
                              ('num', 'passthrough', [c for c in features if c != 'experiment'])])

def make_dummy(features):
    return Pipeline([('prep', prep_scaled(features)), ('clf', DummyClassifier(strategy='most_frequent'))])

def make_lr(features):
    return Pipeline([('prep', prep_scaled(features)), ('clf', LogisticRegression(max_iter=2000, random_state=42))])

def make_dt(features, **params):
    return Pipeline([('prep', prep_trees(features)), ('clf', DecisionTreeClassifier(random_state=42, **params))])

def make_rf(features, **params):
    return Pipeline([('prep', prep_trees(features)),
                     ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, **params))])

def cv_scores(make_model, features, **params):
    rows = []
    for k in sorted(train['fold'].unique()):
        tr, va = train[train['fold'] != k], train[train['fold'] == k]
        m = make_model(features, **params).fit(tr[features], tr['label'])
        p = m.predict(va[features])
        rows.append((accuracy_score(va['label'], p), f1_score(va['label'], p, average='macro')))
    return pd.DataFrame(rows, columns=['accuracy', 'macro_f1'])

## Step 2 - Tuning with participant-wise cross-validation
A tree has settings (**hyperparameters**) that control how complex it can become:
* `max_depth`: how many questions in a row it may ask (deeper = more detail, more overfitting);
* `min_samples_leaf`: the minimum number of trials needed to make a decision (larger = smoother, safer);
* Random Forest only: `max_features`: how many features each split may look at.

We try every combination in a small grid, score each with 5-fold participant-wise CV on the **training** participants, and keep the combination with the best **macro-F1**.

In [ ]:
def tune(make_model, grid, label):
    results = []
    for params in ParameterGrid(grid):
        d = cv_scores(make_model, ALL_FEATURES, **params)
        results.append((params, d))
    table = pd.DataFrame([{'params': str(p), 'macro_f1_mean': d.macro_f1.mean(), 'macro_f1_std': d.macro_f1.std(),
                           'accuracy_mean': d.accuracy.mean()} for p, d in results]
                         ).sort_values('macro_f1_mean', ascending=False).reset_index(drop=True)
    best_params, best_d = max(results, key=lambda r: r[1].macro_f1.mean())
    print(f'--- {label}: top 5 of {len(results)} settings (CV on training participants) ---')
    print(table.head(5).round(3).to_string(index=False))
    print('Best:', best_params, '\n')
    return best_params, best_d

dt_grid = {'max_depth': [3, 5, 8, 12, None], 'min_samples_leaf': [10, 30, 60]}
rf_grid = {'max_depth': [6, 12, None], 'min_samples_leaf': [5, 20, 50], 'max_features': ['sqrt', 0.5]}

best_dt_params, cv_dt = tune(make_dt, dt_grid, 'Decision Tree')
best_rf_params, cv_rf = tune(make_rf, rf_grid, 'Random Forest')

## Step 3 - Fair comparison of all four models
For each model we report:
* **CV macro-F1** (5 participant-wise folds on the training participants; for the trees these are the tuned settings);
* **Test** accuracy, macro-F1 and **severe error rate** on the 30 unseen test participants (Low predicted as High, or the reverse).

Each model is trained on all training participants and tested once.

In [ ]:
def evaluate(model, name):
    p = model.predict(test[ALL_FEATURES])
    pr, rc, f1, _ = precision_recall_fscore_support(test['label'], p, average='macro', zero_division=0)
    return p, {'model': name, 'test_accuracy': accuracy_score(test['label'], p), 'test_precision_macro': pr,
               'test_recall_macro': rc, 'test_f1_macro': f1,
               'severe_error_rate': float((np.abs(test['label'].values - p) == 2).mean())}

models = {
    'Dummy (most frequent)': (make_dummy(ALL_FEATURES), cv_scores(lambda f: make_dummy(f), ALL_FEATURES)),
    'Logistic Regression':   (make_lr(ALL_FEATURES),    cv_scores(lambda f: make_lr(f), ALL_FEATURES)),
    'Decision Tree':         (make_dt(ALL_FEATURES, **best_dt_params), cv_dt),
    'Random Forest':         (make_rf(ALL_FEATURES, **best_rf_params), cv_rf),
}

rows, preds = [], {}
for name, (model, cv_d) in models.items():
    model.fit(train[ALL_FEATURES], train['label'])
    preds[name], res = evaluate(model, name)
    res['cv_macro_f1_mean'] = cv_d.macro_f1.mean()
    res['cv_macro_f1_std'] = cv_d.macro_f1.std()
    res['cv_accuracy_mean'] = cv_d.accuracy.mean()
    rows.append(res)

comparison = pd.DataFrame(rows)[['model', 'cv_macro_f1_mean', 'cv_macro_f1_std', 'cv_accuracy_mean',
                                 'test_accuracy', 'test_precision_macro', 'test_recall_macro',
                                 'test_f1_macro', 'severe_error_rate']].round(3)
print(comparison.to_string(index=False))

## Step 4 - Charts: scores and confusion matrices
The confusion matrix shows, for each true class (rows), what the model predicted (columns). The diagonal is correct.

In [ ]:
real = comparison[comparison['model'] != 'Dummy (most frequent)']
x = np.arange(len(comparison)); w = 0.38
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, comparison['cv_macro_f1_mean'], w, yerr=comparison['cv_macro_f1_std'], label='5-fold CV (train participants)', color='#5b9bd5')
ax.bar(x + w/2, comparison['test_f1_macro'], w, label='Test (30 unseen participants)', color='#d9534f')
ax.set_xticks(x); ax.set_xticklabels(comparison['model'], rotation=12)
ax.set_ylabel('Macro-F1'); ax.set_title('Model comparison'); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison.png'), dpi=150)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, name in zip(axes, ['Logistic Regression', 'Decision Tree', 'Random Forest']):
    ConfusionMatrixDisplay.from_predictions(test['label'], preds[name], display_labels=LABEL_NAMES,
                                            cmap='Blues', ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrices_all.png'), dpi=150)
plt.show()

## Step 5 - Choose the final Model 1
**The choice is made on cross-validation macro-F1, not on the test score**, so the test participants stay an honest, untouched check.

If the best model is only slightly better than a simpler one (difference smaller than the fold-to-fold spread), a simpler model is easier to explain and is just as good in practice. The next cell prints this comparison so we can decide openly.

In [ ]:
cv_rank = comparison[comparison['model'] != 'Dummy (most frequent)'].sort_values('cv_macro_f1_mean', ascending=False)
print(cv_rank[['model', 'cv_macro_f1_mean', 'cv_macro_f1_std', 'test_f1_macro']].to_string(index=False))

best_name = cv_rank.iloc[0]['model']
second = cv_rank.iloc[1]
gap = cv_rank.iloc[0]['cv_macro_f1_mean'] - second['cv_macro_f1_mean']
print(f'\nBest by CV macro-F1: {best_name}')
print(f'Gap to the runner-up ({second["model"]}): {gap:.3f}  (fold std of the best model: {cv_rank.iloc[0]["cv_macro_f1_std"]:.3f})')
if gap < cv_rank.iloc[0]['cv_macro_f1_std']:
    print('The gap is smaller than the fold-to-fold spread: the models are statistically very close.')

print('\nPer-class report on the test participants for the chosen model:')
print(classification_report(test['label'], preds[best_name], target_names=LABEL_NAMES, zero_division=0))

## Step 6 - Which features does the Random Forest rely on?
We use **permutation importance**: shuffle one feature at a time on the test participants and see how much the macro-F1 drops. A big drop means the model depends on that feature. (This is more trustworthy than the built-in tree importance, which favours features with many distinct values.)

In [ ]:
rf_model = models['Random Forest'][0]
imp = permutation_importance(rf_model, test[ALL_FEATURES], test['label'], scoring='f1_macro',
                             n_repeats=10, random_state=42, n_jobs=-1)
imp_s = pd.Series(imp.importances_mean, index=ALL_FEATURES).sort_values()
print((imp_s[::-1]).round(4).to_string())

plt.figure(figsize=(7, 4.2))
plt.barh(imp_s.index, imp_s.values, xerr=pd.Series(imp.importances_std, index=ALL_FEATURES)[imp_s.index].values, color='steelblue')
plt.xlabel('Drop in macro-F1 when the feature is shuffled'); plt.title('Random Forest - permutation importance (test participants)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'rf_permutation_importance.png'), dpi=150)
plt.show()

## Step 7 - Save the final model and the results
The chosen model is saved as `c3_difficulty_model.pkl` (git-ignored; keep it in Drive). The comparison table and a small JSON record of the decision are saved for the report.

In [ ]:
final_model = models[best_name][0]
joblib.dump(final_model, os.path.join(MODELS_DIR, 'c3_difficulty_model.pkl'))
comparison.to_csv(os.path.join(OUTPUT_DIR, 'results_day4_comparison.csv'), index=False)

params_used = {'Decision Tree': best_dt_params, 'Random Forest': best_rf_params}.get(best_name, {})
info = {'chosen_model': best_name, 'chosen_by': 'highest 5-fold participant-wise CV macro-F1 on training participants',
        'hyperparameters': {k: (None if v is None else v) for k, v in params_used.items()},
        'features': ALL_FEATURES, 'label_names': LABEL_NAMES,
        'tercile_cut_points_ms': meta['tercile_cut_points_ms'],
        'cv_macro_f1': float(cv_rank.iloc[0]['cv_macro_f1_mean']), 'test_macro_f1': float(cv_rank.iloc[0]['test_f1_macro'])}
with open(os.path.join(OUTPUT_DIR, 'best_model_info.json'), 'w') as f:
    json.dump(info, f, indent=2)
print('Saved:', os.path.join(MODELS_DIR, 'c3_difficulty_model.pkl'))
print(json.dumps(info, indent=2))

## Summary - Day 4
Read the comparison table above:
* Which model has the highest cross-validation macro-F1, and is the gap bigger than the fold-to-fold spread?
* Do the test scores agree with the CV ranking?
* How do all models compare with the Dummy (about 0.16-0.18)?
* Which features does the Random Forest depend on (permutation importance)?

**Next: Day 5 - synthetic checkpoint -> restart-summary dataset and QLoRA fine-tuning of Qwen (Model 2).**